En esta estapa ya se limpiaron los datos y se comenzará con el modelo de clasificacion. Como en el articulo, se utilizara LightGBM, una libreria de Microsoft especializada en Gradient Boosting.

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import lightgbm as lgb

X = pd.read_csv("../data/features.csv")
CLASES = ["SLSN-I", "SN II", "SN IIn", "SN Ia", "SN Ib/c"] 
FEATS = [c for c in X.columns if c not in ("NAME", "CLASE")]
y = X["CLASE"].map({c: i for i, c in enumerate(CLASES)}).to_numpy()
Xf = X[FEATS].to_numpy()

SEED = 42
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

def crea_modelo():
    return lgb.LGBMClassifier(
        boosting_type="dart", n_estimators=250, max_depth=5, num_leaves=20,
        reg_lambda=5, class_weight="balanced", random_state=SEED, verbose=-1)

pred = np.empty(len(y), dtype=int)
acc, f1 = [], []
for tr, te in skf.split(Xf, y):
    m = crea_modelo().fit(Xf[tr], y[tr])
    pred[te] = m.predict(Xf[te])
    acc.append(accuracy_score(y[te], pred[te]))
    f1.append(f1_score(y[te], pred[te], average="macro"))

def resumen(v):                       
    v = np.sort(v)
    return round(float(np.median(v)), 3), round(float(v[1]), 3), round(float(v[-2]), 3)

print("accuracy (mediana, 2º menor, 2º mayor):", resumen(acc))
print("F1 macro (mediana, 2º menor, 2º mayor):", resumen(f1))

cm = confusion_matrix(y, pred)
comp = cm / cm.sum(axis=1, keepdims=True)
pure = cm / cm.sum(axis=0, keepdims=True)    
print(pd.DataFrame(comp, index=CLASES, columns=CLASES).round(2))
print(pd.DataFrame(pure, index=CLASES, columns=CLASES).round(2))

tabla = pd.DataFrame({
    "completitud_paper": [0.75, 0.72, 0.52, 0.87, 0.80],
    "completitud_mia": comp.diagonal().round(2),
    "pureza_paper": [0.27, 0.87, 0.51, 0.97, 0.28],
    "pureza_mia": pure.diagonal().round(2)}, index=CLASES)
print(tabla)

accuracy (mediana, 2º menor, 2º mayor): (0.827, 0.809, 0.845)
F1 macro (mediana, 2º menor, 2º mayor): (0.556, 0.49, 0.588)
         SLSN-I  SN II  SN IIn  SN Ia  SN Ib/c
SLSN-I     0.33   0.23    0.30   0.12     0.01
SN II      0.03   0.73    0.08   0.10     0.06
SN IIn     0.11   0.26    0.52   0.06     0.06
SN Ia      0.01   0.04    0.01   0.89     0.06
SN Ib/c    0.01   0.14    0.05   0.25     0.55
         SLSN-I  SN II  SN IIn  SN Ia  SN Ib/c
SLSN-I     0.23   0.02    0.10   0.00     0.00
SN II      0.26   0.70    0.30   0.02     0.11
SN IIn     0.20   0.06    0.45   0.00     0.03
SN Ia      0.28   0.18    0.11   0.96     0.58
SN Ib/c    0.02   0.03    0.04   0.01     0.27
         completitud_paper  completitud_mia  pureza_paper  pureza_mia
SLSN-I                0.75             0.33          0.27        0.23
SN II                 0.72             0.73          0.87        0.70
SN IIn                0.52             0.52          0.51        0.45
SN Ia                 0.87       

In [2]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
tabla.to_csv("../results/comparacion_paper_vs_mia.csv")
pd.DataFrame(comp, index=CLASES, columns=CLASES).to_csv("../results/completitud_baseline.csv")
pd.DataFrame(pure, index=CLASES, columns=CLASES).to_csv("../results/pureza_baseline.csv")

In [3]:
def corre_cv(seed, class_weight="balanced"):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    pred_s = np.empty(len(y), dtype=int)
    for tr, te in skf.split(Xf, y):
        m = lgb.LGBMClassifier(boosting_type="dart", n_estimators=250, max_depth=5,
                               num_leaves=20, reg_lambda=5, class_weight=class_weight,
                               random_state=seed, verbose=-1).fit(Xf[tr], y[tr])
        pred_s[te] = m.predict(Xf[te])
    return pred_s

filas = []
for s in range(5):
    p = corre_cv(s)
    cm_s = confusion_matrix(y, p)
    filas.append({"semilla": s,
                  "accuracy": accuracy_score(y, p),
                  "F1_macro": f1_score(y, p, average="macro"),
                  **dict(zip(CLASES, (cm_s.diagonal() / cm_s.sum(axis=1)).round(2)))})
print(pd.DataFrame(filas).round(3))

   semilla  accuracy  F1_macro  SLSN-I  SN II  SN IIn  SN Ia  SN Ib/c
0        0     0.829     0.532    0.22   0.72    0.51   0.89     0.54
1        1     0.829     0.533    0.25   0.72    0.50   0.89     0.52
2        2     0.820     0.548    0.41   0.71    0.50   0.88     0.54
3        3     0.831     0.544    0.28   0.72    0.52   0.90     0.53
4        4     0.830     0.542    0.28   0.72    0.52   0.89     0.52


ModuleNotFoundError: No module named 'imblearn'